# PUMA Training Notebook (Colab + Google Drive)

Notebook nay train theo codebase `PUMA_Nucleus_Segmentation` voi dataset PUMA (giu nguyen cau truc folder goc) va luu checkpoint len Google Drive.

## Muc tieu
- Dung dung kich thuoc input cua challenge (ROI 1024x1024).
- Dung dung so class theo track:
  - Track 1: nuclei 3 class foreground.
  - Track 2: nuclei 10 class foreground.
- Train theo pipeline co san cua du an (`prepare_data.py` + `puma-train`).
- Luu model/checkpoint/log vao Drive de khong mat khi het session Colab.

In [ ]:
# ===== 1) Install dependencies =====
# Neu ban chay local Jupyter thi co the bo qua cell nay.

import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install -U pip
    !pip -q install -e .[dev,cellpose]
else:
    print('Not running in Colab. Skip pip install step if environment da san sang.')

In [ ]:
# ===== 2) Mount Google Drive + set paths =====
from pathlib import Path
import os
import shutil

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/PUMA_Nucleus_Segmentation')
else:
    # Chinh lai path nay neu chay local
    REPO_ROOT = Path.cwd().resolve().parent

# Duong dan dataset goc (giu nguyen cau truc folder cua ban)
DATASET_ROOT = REPO_ROOT / 'dataset_PUMA'

# Noi trung gian de fit voi scripts/prepare_data.py
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
SPLIT_DIR = REPO_ROOT / 'data' / 'splits'

# Thu muc Drive de luu checkpoint/log
DRIVE_ROOT = Path('/content/drive/MyDrive/PUMA_experiments') if IN_COLAB else (REPO_ROOT / 'drive_exports')
CHECKPOINT_DIR = DRIVE_ROOT / 'models'
LOG_DIR = DRIVE_ROOT / 'runs'
OUTPUT_DIR = DRIVE_ROOT / 'outputs'

for p in [RAW_DIR / 'images', RAW_DIR / 'annotations', PROCESSED_DIR, SPLIT_DIR, CHECKPOINT_DIR, LOG_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)

In [ ]:
# ===== 3) Build raw dataset expected by prepare_data.py =====
# prepare_data.py can duoc:
#   data/raw/images/<stem>.tif
#   data/raw/annotations/<stem>.geojson
# Trong dataset_PUMA, nuclei annotation dang la <stem>_nuclei.geojson.

from pathlib import Path
import shutil

img_src_dir = DATASET_ROOT / '01_training_dataset_tif_ROIs'
ann_src_dir = DATASET_ROOT / '01_training_dataset_geojson_nuclei'

assert img_src_dir.exists(), f'Missing image dir: {img_src_dir}'
assert ann_src_dir.exists(), f'Missing annotation dir: {ann_src_dir}'

img_dst_dir = RAW_DIR / 'images'
ann_dst_dir = RAW_DIR / 'annotations'

# Xoa ban cu de dam bao split moi sach (optional)
for old in img_dst_dir.glob('*'):
    old.unlink()
for old in ann_dst_dir.glob('*'):
    old.unlink()

copied = 0
missing_ann = []

for img_path in sorted(img_src_dir.glob('*.tif')):
    stem = img_path.stem
    ann_path = ann_src_dir / f'{stem}_nuclei.geojson'
    if not ann_path.exists():
        missing_ann.append(stem)
        continue

    shutil.copy2(img_path, img_dst_dir / f'{stem}.tif')
    shutil.copy2(ann_path, ann_dst_dir / f'{stem}.geojson')
    copied += 1

print(f'Copied {copied} image-annotation pairs.')
if missing_ann:
    print(f'Missing annotations for {len(missing_ann)} images, example: {missing_ann[:5]}')

print('Raw images:', len(list(img_dst_dir.glob('*.tif'))))
print('Raw annotations:', len(list(ann_dst_dir.glob('*.geojson'))))

In [ ]:
# ===== 4) Set training knobs =====
# Track 1 -> 3 class nuclei foreground
# Track 2 -> 10 class nuclei foreground
TRACK = 1

# Dung dung input size challenge: 1024x1024
# Trong config se set image_size = null (giu nguyen kich thuoc goc)

VAL_SPLIT = 0.15
TEST_SPLIT = 0.10
SEED = 42

# So epoch co the giam khi data subset nho
SEG_EPOCHS = 20
CLS_PHASE1_EPOCHS = 5
CLS_PHASE2_EPOCHS = 15

print({'TRACK': TRACK, 'SEG_EPOCHS': SEG_EPOCHS, 'CLS_PHASE1_EPOCHS': CLS_PHASE1_EPOCHS, 'CLS_PHASE2_EPOCHS': CLS_PHASE2_EPOCHS})

In [ ]:
# Sanity check so class theo track
from puma_seg.data.geojson_parser import NUM_CLASSES

n_fg = NUM_CLASSES[TRACK] - 1
print(f'Track {TRACK}: foreground classes = {n_fg}')
assert (TRACK == 1 and n_fg == 3) or (TRACK == 2 and n_fg == 10), 'Sai cau hinh track/class!'

In [ ]:
# ===== 5) Prepare processed data (mask + label crop-level) =====
import subprocess
import sys

cmd = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / 'prepare_data.py'),
    '--raw-dir', str(RAW_DIR),
    '--out-dir', str(PROCESSED_DIR),
    '--split-dir', str(SPLIT_DIR),
    '--track', str(TRACK),
    '--val-split', str(VAL_SPLIT),
    '--test-split', str(TEST_SPLIT),
    '--seed', str(SEED),
    '--image-ext', '.tif',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

In [ ]:
# ===== 6) Generate a training config that saves checkpoints to Drive =====
import yaml
from datetime import datetime

run_name = f'track{TRACK}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
cfg_path = REPO_ROOT / 'configs' / f'colab_{run_name}.yaml'

cfg = {
    'data': {
        'raw_dir': str(RAW_DIR),
        'processed_dir': str(PROCESSED_DIR),
        'split_file': str(SPLIT_DIR / 'split.json'),
        'track': TRACK,
        'image_size': None,   # IMPORTANT: keep original 1024x1024 input size
        'val_split': VAL_SPLIT,
        'test_split': TEST_SPLIT,
        'crop_size': 64,
    },
    'segmentation': {
        'pretrained_model': 'cyto3',
        'gpu': True,
        'diameter': 17.0,
        'nchan': 2,
        'channels': [0, 0],
        'learning_rate': 1e-5,
        'weight_decay': 0.1,
        'n_epochs': SEG_EPOCHS,
        'batch_size': 4,
        'model_name': f'puma_cellpose_{run_name}',
    },
    'classification': {
        'pretrained': True,
        'freeze_backbone': True,
        'dropout': 0.3,
        'phase1_epochs': CLS_PHASE1_EPOCHS,
        'phase1_lr': 1e-3,
        'phase2_epochs': CLS_PHASE2_EPOCHS,
        'phase2_lr_head': 1e-4,
        'phase2_lr_backbone': 1e-6,
        'unfreeze_groups': 2,
        'weight_decay': 1e-4,
        'label_smoothing': 0.1,
        'patience': 8,
        'batch_size': 32,
        'use_amp': True,
        'model_name': f'best_classifier_{run_name}',
    },
    'paths': {
        'save_dir': str(CHECKPOINT_DIR / run_name),
        'log_dir': str(LOG_DIR / run_name),
        'output_dir': str(OUTPUT_DIR / run_name),
    },
}

with open(cfg_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Config saved to:', cfg_path)
print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
# ===== 7) Train model (segmentation + classification) =====
import subprocess
import sys

train_cmd = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / 'train.py'),
    '--config', str(cfg_path),
    '--mode', 'both',
]

print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, cwd=REPO_ROOT, check=True)
print('Training complete.')

In [ ]:
# ===== 8) Verify checkpoints on Drive =====
from pathlib import Path

save_dir = Path(cfg['paths']['save_dir'])
log_dir = Path(cfg['paths']['log_dir'])

print('Save dir exists:', save_dir.exists(), save_dir)
print('Log dir exists:', log_dir.exists(), log_dir)

if save_dir.exists():
    for p in sorted(save_dir.rglob('*'))[:30]:
        print('-', p)

print('\nDone. Checkpoint da duoc luu len Google Drive.')